In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.data_processor import TennisDataProcessor

In [ ]:
# Load and process data
processor = TennisDataProcessor()
matches = processor.load_matches_data(data_path="../data/raw/")
clean_matches = processor.clean_matches_data()

In [ ]:
# Calculate recent form for all matches
form_df = processor.calculate_recent_form(clean_matches)

print(f"Form calculations completed for {len(form_df)} player-match instances")
print(f"Date range: {form_df['date'].min()} to {form_df['date'].max()}")
print(f"Unique players: {form_df['player'].nunique()}")

In [ ]:
# Test with specific player to validate logic
test_player = "Novak Djokovic"  
player_form = form_df[form_df['player'] == test_player].sort_values('date')

if len(player_form) > 0:
    print(f"=== {test_player.upper()} RECENT FORM ANALYSIS ===")
    print(f"Total form calculations for {test_player}: {len(player_form)}")
    print(f"{'Date':<12} {'Matches':<8} {'Win Rate':<10} {'Weighted':<10}")
    print("-" * 45)
    
    # Show a sample of form progression
    sample_indices = np.linspace(0, len(player_form)-1, min(10, len(player_form)), dtype=int)
    
    for idx in sample_indices:
        row = player_form.iloc[idx]
        print(f"{row['date'].strftime('%Y-%m-%d'):<12} "
              f"{row['recent_matches']:<8} "
              f"{row['win_rate']:<10.3f} "
              f"{row['weighted_win_rate']:<10.3f}")
else:
    print(f"{test_player} not found in data. Available players:")
    print(form_df['player'].value_counts().head())

In [ ]:
print("=== EXPONENTIAL WEIGHTING VALIDATION ===")

if len(player_form) > 0:
    # Check that weighted win rate handles recent vs older matches properly
    recent_high_form = player_form[player_form['weighted_win_rate'] > 0.7]
    recent_low_form = player_form[player_form['weighted_win_rate'] < 0.3]
    
    print(f"High form periods (>70%): {len(recent_high_form)}")
    print(f"Low form periods (<30%): {len(recent_low_form)}")
    
    # Check correlation between win_rate and weighted_win_rate
    correlation = np.corrcoef(player_form['win_rate'], player_form['weighted_win_rate'])[0,1]
    print(f"Correlation between simple and weighted win rates: {correlation:.3f}")
    print("(Should be high but not 1.0, showing exponential weighting effect)")
    
    # Show some examples where weighting makes a difference
    form_diff = np.abs(player_form['win_rate'] - player_form['weighted_win_rate'])
    significant_diff = player_form[form_diff > 0.1]
    
    print(f"\nMatches where weighting changed form by >10%: {len(significant_diff)}")
    if len(significant_diff) > 0:
        print("Sample cases where exponential weighting mattered:")
        for _, row in significant_diff.head(3).iterrows():
            print(f"  {row['date'].strftime('%Y-%m-%d')}: "
                  f"Simple={row['win_rate']:.3f}, Weighted={row['weighted_win_rate']:.3f}")

In [ ]:
print("=== EDGE CASE VALIDATION ===")

# Players with no recent matches should have default 0.5 form
no_recent_matches = form_df[form_df['recent_matches'] == 0]
print(f"Instances with no recent matches: {len(no_recent_matches)}")
print(f"Percentage of total: {len(no_recent_matches)/len(form_df)*100:.1f}%")

if len(no_recent_matches) > 0:
    default_rates = no_recent_matches[['win_rate', 'weighted_win_rate']].drop_duplicates()
    print(f"Default win rates for no recent matches:\n{default_rates}")
    
    # Check if all no-recent-match cases have 0.5 form
    all_default = ((no_recent_matches['win_rate'] == 0.5) & 
                   (no_recent_matches['weighted_win_rate'] == 0.5)).all()
    print(f"All no-recent-match cases have default 0.5 form: {all_default}")

# Check for any invalid values
print(f"\n=== DATA QUALITY VALIDATION ===")
invalid_form = form_df[
    (form_df['win_rate'] < 0) | (form_df['win_rate'] > 1) |
    (form_df['weighted_win_rate'] < 0) | (form_df['weighted_win_rate'] > 1)
]
print(f"Invalid form values (outside [0,1]): {len(invalid_form)}")

if len(invalid_form) > 0:
    print("ERROR: Found invalid form values!")
    print(invalid_form[['player', 'date', 'win_rate', 'weighted_win_rate']])
else:
    print("All form values are valid (between 0 and 1)")

# Check for NaN values
nan_values = form_df[['win_rate', 'weighted_win_rate']].isnull().sum()
print(f"NaN values in win_rate: {nan_values['win_rate']}")
print(f"NaN values in weighted_win_rate: {nan_values['weighted_win_rate']}")

In [ ]:
print("=== EXPONENTIAL DECAY FORMULA VALIDATION ===")

# Validate exponential decay math
test_days = [1, 7, 14, 30, 60, 90]
decay_factor = 0.1

print(f"Decay factor: {decay_factor}")
print(f"Formula: weight = exp(-{decay_factor} * days)")
print(f"{'Days':<5} {'Weight':<8} {'% of Day 1':<12}")
print("-" * 25)

day_1_weight = np.exp(-decay_factor * 1)
for days in test_days:
    weight = np.exp(-decay_factor * days)
    pct_of_day1 = (weight / day_1_weight) * 100
    print(f"{days:<5} {weight:<8.4f} {pct_of_day1:<12.1f}%")

print("\nWeights decrease exponentially with time as expected")
print("Note: After 90 days, matches have ~0.04% the weight of day-1 matches")